<a href="https://colab.research.google.com/github/AllanGiaretta26/Tradutor_Artigos_Tecnicos_Azure/blob/main/tdt_arquivos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import os

os.environ["AZURE_TRANSLATOR_KEY"] = "AZURE_KEY"
os.environ["AZURE_TRANSLATOR_ENDPOINT"] = "AZURE_ENDPOINT"
os.environ["AZURE_TRANSLATOR_REGION"] = "AZURE_REGION"


In [2]:
!pip install python-docx requests


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 16.1 MB/s eta 0:00:00


In [35]:
import os
import requests
import uuid

AZURE_KEY = os.getenv("AZURE_TRANSLATOR_KEY")
AZURE_ENDPOINT = os.getenv("AZURE_TRANSLATOR_ENDPOINT")
AZURE_REGION = os.getenv("AZURE_TRANSLATOR_REGION")

def traduzir_texto(texto, idioma_destino="en"):
    url = f"{AZURE_ENDPOINT}/translate"

    params = {
        "api-version": "3.0",
        "to": idioma_destino
    }

    headers = {
        "Ocp-Apim-Subscription-Key": AZURE_KEY,
        "Ocp-Apim-Subscription-Region": AZURE_REGION,
        "Content-Type": "application/json",
        "X-ClientTraceId": str(uuid.uuid4())
    }

    body = [{"text": texto}]

    response = requests.post(url, params=params, headers=headers, json=body)
    response.raise_for_status()

    resultado = response.json()
    return resultado[0]["translations"][0]["text"]

In [37]:
texto = input("Digite o texto para traduzir: ")
idioma = input("Idioma de destino (ex: en, pt, es, fr): ")

traducao = traduzir_texto(texto, idioma)
print("\nTexto traduzido:")
print(traducao)



Digite o texto para traduzir: Hi
Idioma de destino (ex: en, pt, es, fr): en

Texto traduzido:
Hi


In [72]:
from google.colab import files
import os

uploaded = files.upload()
arquivo_entrada = next(iter(uploaded))

caminho_entrada = f"/content/{arquivo_entrada}"

print("Arquivo original:", caminho_entrada)
print("Conteúdo do diretório:", os.listdir("/content"))



Saving tdt_documento.docx to tdt_documento.docx
Arquivo original: /content/tdt_documento.docx
Conteúdo do diretório: ['.config', '.ipynb_checkpoints', 'tdt_documento.docx', 'sample_data']


In [73]:
from docx import Document
import os

def traduzir_docx(caminho_entrada, idioma_destino="en"):
    if not os.path.exists(caminho_entrada):
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho_entrada}")

    doc = Document(caminho_entrada)
    novo_doc = Document()

    for p in doc.paragraphs:
        texto = p.text.strip()
        if texto:
            texto_traduzido = traduzir_texto(texto, idioma_destino)
            novo_doc.add_paragraph(texto_traduzido)
        else:
            novo_doc.add_paragraph("")

    caminho_saida = f"/content/traduzido_{idioma_destino}.docx"
    novo_doc.save(caminho_saida)

    return caminho_saida


In [74]:
idioma = input("Idioma de destino (ex: en, es, fr, pt): ")

arquivo_traduzido = traduzir_docx(caminho_entrada, idioma)

print("Arquivo gerado em:", arquivo_traduzido)
print("Conteúdo atual do diretório:", os.listdir("/content"))


Idioma de destino (ex: en, es, fr): pt
Arquivo gerado em: /content/traduzido_pt.docx
Conteúdo atual do diretório: ['.config', '.ipynb_checkpoints', 'traduzido_pt.docx', 'tdt_documento.docx', 'sample_data']
